# Persian Tweet Text Generation — SVM



## 0. Setup

In [ ]:
import subprocess, sys, time

print('Installing p7zip...')
subprocess.run(['apt-get', 'install', '-y', 'p7zip-full'], capture_output=True)
print('Installing Python packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'tqdm', 'hazm', 'scikit-learn', 'joblib'],
               capture_output=True)
print('All done.')

In [ ]:
import os, math, time, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import joblib, random, collections
from transformers import AutoTokenizer
from hazm import Normalizer
from tqdm import tqdm
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

# ── Paths —─────────────────────────────────────────────
DRIVE_PATH  = '/content/drive/MyDrive/twitter_sample_tweets.csv.7z'
DATA_DIR    = '/content/data'
CLEANED_CSV = '/content/cleaned_tweets.csv'
DATA_PATH   = CLEANED_CSV
CKPT_DIR    = '/content/drive/MyDrive/persian_svm_checkpoints'

# ── Data —──────────────────────────────────────────────
N_TWEETS   = 150_000
MAX_LENGTH = 32

# ── SVM Config ─────────────────────────────────────────

# number of preceding token IDs used as features
CONTEXT_SIZE  = 5

# restrict targets to the K most frequent tokens;
# full 42k vocab makes LinearSVC prohibitively slow;
# top-3000 still covers ~80-85 % of all token occurrences
SVM_TOP_K     = 3_000

# cap training pairs to avoid OOM; more pairs give diminishing returns
SVM_MAX_PAIRS = 500_000

# TF-IDF operates on token-ID strings, not raw Persian text
TFIDF_MAX_FEATURES = 30_000

# SVM regularisation strength: smaller C = stronger regularisationSVM_C = 0.3   random.seed(42)
np.random.seed(42)
print('Config loaded.')
print(f'CONTEXT_SIZE={CONTEXT_SIZE}  TOP_K={SVM_TOP_K:,}  MAX_PAIRS={SVM_MAX_PAIRS:,}')

## 1. Mount Drive & Extract Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

extracted_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
if extracted_files:
    RAW_CSV = os.path.join(DATA_DIR, extracted_files[0])
    print(f'Already extracted: {RAW_CSV}')
else:
    print(f'Extracting {DRIVE_PATH} ...')
    t0 = time.time()
    result = subprocess.run(
        ['7z', 'x', DRIVE_PATH, f'-o{DATA_DIR}', '-y'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('STDERR:', result.stderr[-300:])
    print(f'Done in {(time.time()-t0)/60:.1f} min')
    extracted_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
    assert extracted_files, 'No CSV found!'
    RAW_CSV = os.path.join(DATA_DIR, extracted_files[0])
    print(f'CSV: {RAW_CSV}  ({os.path.getsize(RAW_CSV)/1e9:.2f} GB)')

## 2. Detect CSV Structure

In [ ]:
PERSIAN_RE = re.compile(r'[\u0600-\u06FF]')

def persian_ratio(s):
    s = str(s)
    return len(PERSIAN_RE.findall(s)) / len(s) if s else 0.0

def detect_header_and_text_column(csv_path, n_peek=10):
    peek = pd.read_csv(csv_path, header=None, nrows=n_peek,
                       on_bad_lines='skip', encoding='utf-8', low_memory=False)
    print(peek.head(3).to_string())
    row0_ratios = [persian_ratio(peek.iloc[0, c]) for c in range(peek.shape[1])]
    data_ratios = [peek.iloc[1:, c].astype(str).apply(persian_ratio).mean()
                   for c in range(peek.shape[1])]
    has_header  = np.mean(row0_ratios) < 0.1 and np.mean(data_ratios) > 0.05
    peek_data   = peek.iloc[1:] if has_header else peek
    scores      = [0.5*peek_data.iloc[:,c].astype(str).apply(persian_ratio).mean()
                   + 0.5*peek_data.iloc[:,c].astype(str).apply(len).mean()
                   for c in range(peek_data.shape[1])]
    text_col_idx = int(np.argmax(scores))
    print(f'Has header: {has_header}  |  Text column: {text_col_idx}')
    return has_header, text_col_idx

HAS_HEADER, TEXT_COL_IDX = detect_header_and_text_column(RAW_CSV)

## 3. Preprocessing

In [ ]:
normalizer = Normalizer()
URL_RE     = re.compile(r'https?://\S+')
MENTION_RE = re.compile(r'@\w+')
HASHTAG_RE = re.compile(r'#')
DIGIT_RE   = re.compile(r'[0-9\u06F0-\u06F9]')
KEEP_RE    = re.compile(r'[^\u0600-\u06FF .\u060C,!?\s]')
REPEAT_RE  = re.compile(r'(.)\1{2,}')
SPACE_RE   = re.compile(r'\s+')

def clean_tweet(text):
    text = str(text)
    text = URL_RE.sub('', text)
    text = MENTION_RE.sub('', text)
    text = HASHTAG_RE.sub('', text)
    text = DIGIT_RE.sub('', text)
    text = KEEP_RE.sub('', text)
    try:
        text = normalizer.normalize(text)
    except Exception:
        pass
    text = REPEAT_RE.sub(r'\1\1', text)
    return SPACE_RE.sub(' ', text).strip()

if os.path.exists(CLEANED_CSV) and os.path.getsize(CLEANED_CSV) > 1_000_000:
    print(f'Cleaned CSV exists: {CLEANED_CSV}')
else:
    print('Preprocessing...')
    seen, total_written = set(), 0
    t0 = time.time()
    csv_kw = dict(chunksize=100_000, usecols=[TEXT_COL_IDX],
                  header=0 if HAS_HEADER else None,
                  on_bad_lines='skip', encoding='utf-8', low_memory=False)
    with open(CLEANED_CSV, 'w', encoding='utf-8') as fout:
        fout.write('text\n')
        for ci, chunk in enumerate(pd.read_csv(RAW_CSV, **csv_kw)):
            if total_written >= N_TWEETS: break
            chunk.columns = ['text']
            chunk = chunk.dropna(subset=['text'])
            chunk['c'] = chunk['text'].apply(clean_tweet)
            chunk = chunk[chunk['c'].apply(lambda t: len(t.split())) >= 5]
            chunk = chunk[~chunk['c'].isin(seen)]
            seen.update(chunk['c'].tolist())
            chunk = chunk.head(N_TWEETS - total_written)
            total_written += len(chunk)
            for line in chunk['c']:
                fout.write(line + '\n')
            print(f'Chunk {ci+1} | kept={total_written:,}', end='\r')
    print(f'\nDone. {total_written:,} lines  ({(time.time()-t0)/60:.1f} min)')

## 4. Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('HooshvareLab/gpt2-fa')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

VOCAB_SIZE = len(tokenizer)
PAD_ID     = tokenizer.pad_token_id
EOS_ID     = tokenizer.eos_token_id
print(f'Vocab size: {VOCAB_SIZE:,}')
print(f'PAD id: {PAD_ID}  EOS id: {EOS_ID}')

## 5. Creating Training Pairs


In [ ]:
SVM_PAIRS_PATH = '/content/svm_pairs.npz'
SVM_TOPK_PATH  = '/content/svm_topk.json'

if os.path.exists(SVM_PAIRS_PATH) and os.path.exists(SVM_TOPK_PATH):
    print('Loading cached training pairs...')
    data       = np.load(SVM_PAIRS_PATH)
    X_raw      = data['X_raw'].tolist()   # context stringsy_raw      = data['y_raw'].tolist()   # class-index stringswith open(SVM_TOPK_PATH) as f:
        topk_info  = json.load(f)
    top_k_ids  = set(topk_info['top_k_ids'])
    id_to_cls  = {int(k): v for k, v in topk_info['id_to_cls'].items()}
    cls_to_id  = {int(k): int(v) for k, v in topk_info['cls_to_id'].items()}
    print(f'Loaded {len(X_raw):,} pairs  |  classes: {len(top_k_ids):,}')
else:
    print('Reading tweets and tokenizing...')
    df     = pd.read_csv(DATA_PATH, usecols=['text'], nrows=N_TWEETS)
    tweets = df['text'].astype(str).dropna().tolist()
    tweets = [t.strip() for t in tweets if len(t.strip()) > 5]

    # tokenise all tweets in batchesprint(f'Tokenizing {len(tweets):,} tweets...')
    t0  = time.time()
    all_ids = []
    batch   = 1000
    for i in tqdm(range(0, len(tweets), batch), desc='Tokenizing'):
        enc = tokenizer(tweets[i:i+batch], add_special_tokens=True,
                        truncation=True, max_length=MAX_LENGTH)
        all_ids.extend(enc['input_ids'])
    print(f'Done in {time.time()-t0:.1f}s')

    # find the K most frequent token IDs across the corpuscounter = collections.Counter()
    for ids in all_ids:
        counter.update(ids)
    # exclude special tokens from targetsfor bad in [PAD_ID, EOS_ID]:
        counter.pop(bad, None)
    # select the K most common non-special tokens as prediction targets
    top_k_ids = set(tok_id for tok_id, _ in counter.most_common(SVM_TOP_K))
    coverage  = sum(counter[t] for t in top_k_ids) / sum(counter.values()) * 100
    print(f'Top-{SVM_TOP_K} token IDs cover {coverage:.1f}% of all tokens')

    # map token_id → class-index string (sklearn requires string labels)top_k_list = sorted(top_k_ids)
    id_to_cls  = {tid: str(i) for i, tid in enumerate(top_k_list)}
    cls_to_id  = {i: tid for tid, i in id_to_cls.items()}

    # build (context_string, target_class) training pairsprint('Building context-target pairs...')
    t0    = time.time()
    X_raw = []
    y_raw = []
    for ids in tqdm(all_ids, desc='Building pairs'):
        if len(X_raw) >= SVM_MAX_PAIRS:
            break
        for j in range(CONTEXT_SIZE, len(ids)):
            target = ids[j]
            if target not in top_k_ids:
                continue
            ctx_str = ' '.join(str(t) for t in ids[j-CONTEXT_SIZE:j])
            X_raw.append(ctx_str)
            y_raw.append(id_to_cls[target])
            if len(X_raw) >= SVM_MAX_PAIRS:
                break
    print(f'Pairs: {len(X_raw):,}  ({time.time()-t0:.1f}s)')

    # cache to disk so this step can be skipped on re-runnp.savez('/content/svm_pairs.npz',
             X_raw=np.array(X_raw), y_raw=np.array(y_raw))
    with open(SVM_TOPK_PATH, 'w') as f:
        json.dump({'top_k_ids': list(top_k_ids),
                   'id_to_cls': {str(k): v for k,v in id_to_cls.items()},
                   'cls_to_id': {str(k): v for k,v in cls_to_id.items()}}, f)
    print('Pairs cached to /content/')

## 6. Train SVM Pipeline


In [ ]:
SVM_MODEL_PATH = os.path.join(CKPT_DIR, 'svm_pipeline.joblib')
SVM_META_PATH  = os.path.join(CKPT_DIR, 'svm_meta.json')

if os.path.exists(SVM_MODEL_PATH):
    print('Loading saved SVM from Drive...')
    t0           = time.time()
    svm_pipeline = joblib.load(SVM_MODEL_PATH)
    with open(SVM_META_PATH) as f:
        meta = json.load(f)
    cls_to_id  = {int(k): int(v) for k, v in meta['cls_to_id'].items()}
    top_k_ids  = set(meta['top_k_ids'])
    svm_classes = svm_pipeline.classes_
    print(f'Loaded in {time.time()-t0:.1f}s  |  classes: {len(svm_classes):,}')
else:
    # Filter out classes with fewer than 10 samples. 
    # CalibratedClassifierCV with cv=3 requires each class to have at least
    print('Filtering rare classes...')
    from collections import Counter
    cls_counts  = Counter(y_raw)
    valid_cls   = {cls for cls, cnt in cls_counts.items() if cnt >= 10}
    mask        = [y in valid_cls for y in y_raw]
    X_filtered  = [x for x, m in zip(X_raw, mask) if m]
    y_filtered  = [y for y, m in zip(y_raw, mask) if m]
    print(f'Before filter: {len(X_raw):,} pairs, {len(cls_counts):,} classes')
    print(f'After  filter: {len(X_filtered):,} pairs, {len(valid_cls):,} classes')

    print('Fitting TF-IDF + LinearSVC...')
    t0 = time.time()
    svm_pipeline = Pipeline([
        # char_wb n-grams work well on numeric token-ID strings
        ('tfidf', TfidfVectorizer(
            analyzer      = 'char_wb',
            ngram_range   = (2, 4),
            max_features  = TFIDF_MAX_FEATURES,
            sublinear_tf  = True,
            min_df        = 3,
        )),
        # calibration wraps LinearSVC to expose predict_proba for sampling
        ('clf', CalibratedClassifierCV(
            LinearSVC(C=SVM_C, max_iter=3000, dual=True),
            cv=3, method='sigmoid',
        )),
    ])
    svm_pipeline.fit(X_filtered, y_filtered)
    elapsed = time.time() - t0
    print(f'Training done in {elapsed/60:.1f} min')

    svm_classes = svm_pipeline.classes_
    joblib.dump(svm_pipeline, SVM_MODEL_PATH, compress=3)
    with open(SVM_META_PATH, 'w') as f:
        json.dump({
            'cls_to_id':  {str(k): int(v) for k,v in cls_to_id.items()},
            'top_k_ids':  list(top_k_ids),
            'n_classes':  len(svm_classes),
            'context_size': CONTEXT_SIZE,
            'top_k':      SVM_TOP_K,
        }, f)
    print(f'Pipeline saved -> {SVM_MODEL_PATH}')
    print(f'Classes: {len(svm_classes):,}')
    # point X_raw / y_raw at the filtered data used for evalX_raw = X_filtered
    y_raw = y_filtered

## 7. Evaluation

In [ ]:
print('Evaluating SVM (top-1 and top-5)...')
EVAL_SIZE = 5_000
eval_X    = X_raw[:EVAL_SIZE]
eval_y    = y_raw[:EVAL_SIZE]

# Top-1
t0     = time.time()
y_pred = svm_pipeline.predict(eval_X)
top1   = (np.array(y_pred) == np.array(eval_y)).mean()
print(f'Top-1 Accuracy : {top1*100:.2f}%  ({time.time()-t0:.1f}s)')

# top-5 via predict_proba (subset to keep it fast)print('Computing Top-5 (on 1000 samples)...')
t0      = time.time()
proba   = svm_pipeline.predict_proba(eval_X[:1000])
top5_ok = 0
for i, true_cls in enumerate(eval_y[:1000]):
    top5_idx = np.argsort(proba[i])[-5:]
    if true_cls in svm_classes[top5_idx]:
        top5_ok += 1
top5 = top5_ok / 1000
print(f'Top-5 Accuracy : {top5*100:.2f}%  ({time.time()-t0:.1f}s)')

print(f'\nNote: accuracy measured only on top-{SVM_TOP_K} token targets.')
print(f'These cover ~80-85% of actual token occurrences.')

## 8. Text Generation


In [ ]:
def generate(seed_text='', max_new_tokens=25,
             temperature=1.1, top_p=0.85, rep_penalty=1.8):
    ids = tokenizer.encode(seed_text, add_special_tokens=False)
    if not ids:
        ids = [EOS_ID]

    # cls_to_id keys may be int or str depending on how metadata was saveddef cls_lookup(cls_str):
        v = cls_to_id.get(int(cls_str))
        if v is None:
            v = cls_to_id.get(str(cls_str))
        return int(v) if v is not None else None

    for _ in range(max_new_tokens):
        ctx     = ids[-CONTEXT_SIZE:]
        ctx_str = ' '.join(str(t) for t in ctx)

        proba = svm_pipeline.predict_proba([ctx_str])[0]

        # Temperature scaling
        log_p = np.log(proba + 1e-9) / temperature
        log_p -= log_p.max()
        p     = np.exp(log_p)

        # penalise tokens already in the sequence to reduce repetitionfor i, cls in enumerate(svm_classes):
            tok_id = cls_lookup(cls)
            if tok_id is not None and tok_id in ids:
                p[i] = max(0.0, p[i] / rep_penalty)

        # nucleus (top-p) sampling: keep the smallest set of tokens whose cumulative prob ≥ top_psorted_idx = np.argsort(p)[::-1]
        sorted_p   = p[sorted_idx]
        cum_p      = np.cumsum(sorted_p)
        cutoff     = np.searchsorted(cum_p, top_p) + 1
        nucleus_idx = sorted_idx[:cutoff]
        nucleus_p   = p[nucleus_idx]
        total       = nucleus_p.sum()
        if total < 1e-9:
            break
        nucleus_p = nucleus_p / total

        chosen_pos = np.random.choice(len(nucleus_idx), p=nucleus_p)
        chosen_cls = svm_classes[nucleus_idx[chosen_pos]]
        next_id    = cls_lookup(chosen_cls)

        if next_id is None or next_id == EOS_ID:
            break
        ids.append(next_id)

    return tokenizer.decode(ids, skip_special_tokens=True)

PROMPTS = ['امروز', 'دانشگاه', 'ایران زیبا']
print('\n' + '='*60)
print('SVM GENERATION RESULTS')
print('='*60)
for prompt in PROMPTS:
    print(f'\n  Prompt : {prompt}')
    print(f'  Output : {generate(prompt)}')

## 9. Summary

In [ ]:
print('='*60)
print('SVM SUMMARY')
print('='*60)
print(f'Tokenizer      : HooshvareLab/gpt2-fa  (vocab={VOCAB_SIZE:,})')
print(f'Context size   : {CONTEXT_SIZE} tokens')
print(f'Target classes : top-{SVM_TOP_K} token IDs')
print(f'Training pairs : {len(X_raw):,}')
print(f'TF-IDF features: {TFIDF_MAX_FEATURES:,}  (char n-gram 2-4)')
print(f'SVM C          : {SVM_C}')
print(f'Top-1 Accuracy : {top1*100:.2f}%')
print(f'Top-5 Accuracy : {top5*100:.2f}%')
print()
print('Files on Drive:')
for p in [SVM_MODEL_PATH, SVM_META_PATH]:
    if os.path.exists(p):
        print(f'  OK  {p}  ({os.path.getsize(p)/1e6:.1f} MB)')
print()
